In [ ]:
project_id = 'oceanic-citadel-454608-d2'
from google.cloud import bigquery
client = bigquery.Client(project=project_id)

In [ ]:
import pandas as pd
import numpy as np

# ── Date anchor ──
# AS_OF_DATE = pd.Timestamp.now().normalize() - pd.Timedelta(days=2)
AS_OF_DATE = pd.Timestamp('2026-06-08')

# ── Populations (App not launched on LS yet) ──
POPULATIONS = ['Web', 'Affiliate']

# ── Trim variants per population ──
TRIM_VARIANTS = {
    'Web':       [('winsor', 0.0), ('winsor', 0.01), ('cohort_trim', 0.02)],
    'Affiliate': [('winsor', 0.01)],
}

# ── Patch windows (s, e): growth from ARPU_s to ARPU_e ──
PATCHES = (
    (1, 7), (7, 14), (14, 30), (30, 60), (60, 90),
    (90, 120), (120, 150), (150, 180), (180, 270), (270, 365),
)

# ── Goal horizons (used only for SQL floor-date math) ──
GOAL_HORIZONS = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]

# ── CV thresholds (LS-specific: looser than RP's 0.15) ──
CV_THRESHOLD        = 0.175
CV_GOOD_ENOUGH      = 0.10
MAX_REMOVE_FRACTION = 0.15

# ── Lookback window ──
LOOKBACK_COHORTS = 35

# ── Data-quality guard (LS-specific) ──
MIN_COHORT_DATES = 20

print(f'Config loaded. as_of_date = {AS_OF_DATE.date()}')
print(f'POPULATIONS: {POPULATIONS}')
for pop, variants in TRIM_VARIANTS.items():
    print(f'  {pop}: {variants}')

In [ ]:
from pandas_gbq import read_gbq

SQL_FLOOR_DATE = (
    AS_OF_DATE - pd.Timedelta(days=max(GOAL_HORIZONS) + LOOKBACK_COHORTS + 5)
).date()
print(f'SQL floor date: {SQL_FLOOR_DATE}  (AS_OF_DATE = {AS_OF_DATE.date()})')

users_df = read_gbq(f"""
SELECT
  id,
  CASE
    WHEN affid IN (63, 4432, 4551, 4698, 5048, 5125, 7120, 7253, 7260, 8331, 8345) THEN 'Web'
    -- WHEN affid = 1 THEN 'App'   -- uncomment when LS App launches
    WHEN affid IN (64, 71)  THEN 'PPC'
    WHEN affid IN (0, 78)   THEN 'Organic'
    ELSE 'Affiliate'
  END AS population,
  DATE(MIN(cost_date)) AS cost_date
FROM `analytics.lonestar_cost_per_user`
WHERE cost_date >= DATE('{SQL_FLOOR_DATE}')
  AND affid NOT IN (4866, 7127)
  AND id > 0
GROUP BY 1, 2
""", project_id=project_id, use_bqstorage_api=True)

revenue_df = read_gbq(f"""
SELECT
  playerId AS playerid,
  DATE(date) AS date,
  SUM(amount) / 100.0 AS amount
FROM `lonestar.casino_astropay_dmn`
WHERE Status = 'APPROVED'
  AND date >= DATE('{SQL_FLOOR_DATE}')
GROUP BY 1, 2
""", project_id=project_id, use_bqstorage_api=True)

print(f'users_df:   {len(users_df):,} rows')
print(f'revenue_df: {len(revenue_df):,} rows')
print(users_df['population'].value_counts().to_string())

In [ ]:
def weighted_mean_std_cv(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[m], w[m]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    mu  = np.average(x, weights=w)
    var = np.average((x - mu) ** 2, weights=w)
    sd  = np.sqrt(var)
    cv  = sd / mu if mu != 0 else np.nan
    return mu, sd, cv


def build_user_revenue_cums(users_df, revenue_df, *, max_day=365):
    u = users_df[['id', 'population', 'cost_date']].copy()
    u['population'] = u['population'].astype(str).str.strip()
    u['cost_date']  = pd.to_datetime(u['cost_date'], errors='coerce').dt.date
    u = u.loc[pd.notna(u['cost_date'])].copy()
    u = u.groupby(['population', 'id'], as_index=False)['cost_date'].min()
    u = u.rename(columns={'id': '__uid__'})

    r = revenue_df[['playerid', 'date', 'amount']].copy()
    r['date'] = pd.to_datetime(r['date'], errors='coerce').dt.date
    r = r.loc[pd.notna(r['date'])].copy()

    rr = r.merge(u, left_on='playerid', right_on='__uid__', how='inner')
    rr['dsi'] = (pd.to_datetime(rr['date']) - pd.to_datetime(rr['cost_date'])).dt.days
    rr = rr.loc[(rr['dsi'] >= 0) & (rr['dsi'] <= (max_day - 1))].copy()

    daily_user = (
        rr.groupby(['population', 'cost_date', '__uid__', 'dsi'], observed=True)['amount']
          .sum().reset_index()
          .sort_values(['population', 'cost_date', '__uid__', 'dsi'])
    )
    daily_user['cum_amount'] = (
        daily_user.groupby(['population', 'cost_date', '__uid__'], observed=True)['amount']
                  .cumsum()
    )
    return u, daily_user


print('Math + base table helpers defined.')

In [ ]:
def compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=0.01):
    if top_pct <= 0:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    per_user['cap_e'] = (
        per_user.groupby(['population', 'cost_date'], observed=True)['cum_e']
                .transform(lambda s: s[s > 0].quantile(1.0 - top_pct) if (s > 0).any() else np.inf)
    )
    return per_user[['population', 'cost_date', '__uid__', 'cap_e']]


def apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=0.10):
    if trim_pct <= 0:
        return cohort_users.copy()
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        return cohort_users.copy()
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    depositors = per_user.loc[per_user['cum_e'] > 0].copy()
    if depositors.empty:
        return cohort_users.copy()
    thresholds = (
        depositors.groupby(['population', 'cost_date'], observed=True)['cum_e']
                  .quantile(1.0 - trim_pct).reset_index(name='threshold')
    )
    per_user = per_user.merge(thresholds, on=['population', 'cost_date'], how='left')
    per_user['threshold'] = per_user['threshold'].fillna(np.inf)
    keep = per_user.loc[
        (per_user['cum_e'] == 0) | (per_user['cum_e'] <= per_user['threshold'])
    ][['population', 'cost_date', '__uid__']]
    return keep.copy()


def get_trimmed_for_variant(cohort_users, daily_user_cums, e, method, pct):
    caps, trimmed = None, cohort_users
    if method == 'winsor':
        caps = compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=pct)
    elif method == 'cohort_trim':
        trimmed = apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=pct)
    return trimmed, caps


def sum_cum_at_idx(daily_user_cums, *, cohort_users, idx, caps=None):
    grp_cols = ['population', 'cost_date']
    if idx < 0:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= idx].copy()
    if du.empty:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    per_user = (
        du.groupby(grp_cols + ['__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum')
    )
    per_user = cohort_users.merge(per_user, on=grp_cols + ['__uid__'], how='left')
    per_user['cum'] = per_user['cum'].fillna(0.0)
    if caps is not None:
        per_user = per_user.merge(caps, on=grp_cols + ['__uid__'], how='left')
        per_user['cap_e'] = per_user['cap_e'].fillna(np.inf)
        per_user['cum']   = np.minimum(per_user['cum'], per_user['cap_e'])
    sums = per_user.groupby(grp_cols, observed=True)['cum'].sum().reset_index(name='sum_cum')
    return sums


print('Trim + summation helpers defined.')

In [ ]:
def patch_cv_adaptive_variant(
    u_base, daily_user_cums, *,
    population, s, e, as_of_date, method, pct,
    excluded_uids=None,
    lookback_cohorts=LOOKBACK_COHORTS,
    cv_threshold=CV_THRESHOLD,
    cv_good_enough=CV_GOOD_ENOUGH,
    max_remove_fraction=MAX_REMOVE_FRACTION,
    debug=True,
):
    as_of_date   = pd.to_datetime(as_of_date).normalize()
    cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
    cohort_start = (as_of_date - pd.Timedelta(days=e + (lookback_cohorts - 1))).date()

    cohort_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= cohort_start) &
        (u_base['cost_date'] <= cohort_end)
    ][['population', 'cost_date', '__uid__']].copy()

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    all_cohort_users = cohort_users.copy()
    n_users_in_cohort = int(cohort_users['__uid__'].nunique())

    if excluded_uids:
        cohort_users = cohort_users.loc[
            ~cohort_users['__uid__'].isin(excluded_uids)
        ].copy()

    n_users_after_prior = int(cohort_users['__uid__'].nunique())
    n_users_excluded_prior = n_users_in_cohort - n_users_after_prior

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    n_users_pre_trim = n_users_after_prior

    trimmed_users, caps = get_trimmed_for_variant(
        cohort_users, daily_user_cums, e, method, pct
    )
    n_users_post_trim = int(trimmed_users['__uid__'].nunique())

    newly_excluded = (
        set(cohort_users['__uid__'].unique()) - set(trimmed_users['__uid__'].unique())
    )

    denom_w = (
        trimmed_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                     .nunique().reset_index(name='N_users')
    )
    sum_s = sum_cum_at_idx(
        daily_user_cums, cohort_users=trimmed_users, idx=s - 1, caps=caps
    ).rename(columns={'sum_cum': 'sum_cum_s'})
    sum_e = sum_cum_at_idx(
        daily_user_cums, cohort_users=trimmed_users, idx=e - 1, caps=caps
    ).rename(columns={'sum_cum': 'sum_cum_e'})

    sum_e_all = sum_cum_at_idx(
        daily_user_cums, cohort_users=all_cohort_users, idx=e - 1, caps=None
    ).rename(columns={'sum_cum': 'sum_cum_e_all'})
    total_rev_before_trim = float(sum_e_all['sum_cum_e_all'].sum())

    patch = (
        denom_w
        .merge(sum_s, on=['population', 'cost_date'])
        .merge(sum_e, on=['population', 'cost_date'])
    )
    patch['ARPU_s']       = patch['sum_cum_s'] / patch['N_users']
    patch['ARPU_e']       = patch['sum_cum_e'] / patch['N_users']
    patch['growth_ratio'] = np.where(
        patch['ARPU_s'] > 0, patch['ARPU_e'] / patch['ARPU_s'], np.nan
    )

    _, _, cv_before = weighted_mean_std_cv(patch['growth_ratio'].values, patch['sum_cum_s'].values)

    mu_unw = np.nanmean(patch['growth_ratio'].values)
    patch['abs_dev'] = (patch['growth_ratio'] - mu_unw).abs()
    sorted_dates  = patch.sort_values('abs_dev', ascending=False)['cost_date'].tolist()
    max_removable = max(1, int(np.floor(len(patch) * max_remove_fraction)))

    removed   = []
    remaining = patch.copy()

    for candidate in sorted_dates:
        _, _, cv_now = weighted_mean_std_cv(
            remaining['growth_ratio'].values, remaining['sum_cum_s'].values
        )
        if np.isnan(cv_now) or cv_now <= cv_good_enough:
            break
        if len(removed) >= max_removable:
            break
        removed.append(candidate)
        remaining = remaining.loc[~remaining['cost_date'].isin(removed)]

    mean_a, _, cv_after = weighted_mean_std_cv(
        remaining['growth_ratio'].values, remaining['sum_cum_s'].values
    )
    flagged = (not np.isnan(cv_after)) and (cv_after > cv_threshold)
    total_rev_after_trim = float(patch['sum_cum_e'].sum())

    if debug:
        flag_tag = f'  >>> FLAGGED (cv={cv_after:.4f} > {cv_threshold})' if flagged else ''
        print(
            f'  [{method}_{pct:g}] {s}->{e}  '
            f'cv {cv_before:.4f}->{cv_after:.4f}  '
            f'removed={len(removed)}/{len(patch)}  '
            f'excl_prior={n_users_excluded_prior:,}  '
            f'pre/post={n_users_pre_trim:,}/{n_users_post_trim:,}  '
            f'newly_excl={len(newly_excluded):,}{flag_tag}'
        )

    stats = dict(
        population              = population,
        patch                   = f'{s}->{e}',
        cohort_start            = str(cohort_start),
        cohort_end              = str(cohort_end),
        n_cohort_dates_total    = int(len(patch)),
        n_cohort_dates_kept     = int(len(remaining)),
        n_users_excluded_prior  = n_users_excluded_prior,
        n_users_pre_trim        = n_users_pre_trim,
        n_users_post_trim       = n_users_post_trim,
        n_users_dropped_by_trim = n_users_pre_trim - n_users_post_trim,
        total_rev_before_trim   = total_rev_before_trim,
        total_rev_after_trim    = total_rev_after_trim,
        cv_before               = float(cv_before) if not np.isnan(cv_before) else None,
        cv_after                = float(cv_after)  if not np.isnan(cv_after)  else None,
        mean_after              = float(mean_a)    if not np.isnan(mean_a)    else None,
        flagged                 = bool(flagged),
        removed_dates           = removed,
    )
    return patch, stats, removed, flagged, newly_excluded


print('Variant-aware adaptive CV function defined (persistent-trim, LS).')

In [ ]:
def build_curve_variant(
    u_base, daily_user_cums, *,
    population, as_of_date, method, pct,
    lookback_cohorts=LOOKBACK_COHORTS,
    cv_threshold=CV_THRESHOLD, cv_good_enough=CV_GOOD_ENOUGH,
    max_remove_fraction=MAX_REMOVE_FRACTION, debug=True,
):
    as_of_date = pd.to_datetime(as_of_date).normalize()

    cv_rows      = []
    effective    = []
    excluded_uids = set()

    for (s, e) in PATCHES:
        patch, stats, removed, flagged, newly_excluded = patch_cv_adaptive_variant(
            u_base, daily_user_cums,
            population=population, s=s, e=e,
            as_of_date=as_of_date, method=method, pct=pct,
            excluded_uids=excluded_uids,
            lookback_cohorts=lookback_cohorts,
            cv_threshold=cv_threshold, cv_good_enough=cv_good_enough,
            max_remove_fraction=max_remove_fraction, debug=debug,
        )
        excluded_uids |= newly_excluded

        if not stats:
            continue
        cv_rows.append(stats)
        effective.append({
            's': s, 'e': e,
            'removed_dates': removed,
            'excluded_snapshot': frozenset(excluded_uids),
        })

    if not effective:
        return pd.DataFrame(cv_rows), pd.DataFrame()

    if debug:
        print(f'  Total unique users excluded across all patches: {len(excluded_uids):,}')

    step_rows = []
    for ep in effective:
        s, e      = ep['s'], ep['e']
        bad_dates = set(ep['removed_dates'])
        excl      = ep['excluded_snapshot']
        start_k   = 2 if s == 1 else (s + 1)

        cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
        cohort_start = (as_of_date - pd.Timedelta(days=e + (lookback_cohorts - 1))).date()

        cohort_users = u_base.loc[
            (u_base['population'] == population) &
            (u_base['cost_date'] >= cohort_start) &
            (u_base['cost_date'] <= cohort_end)
        ][['population', 'cost_date', '__uid__']].copy()

        if bad_dates:
            cohort_users = cohort_users.loc[~cohort_users['cost_date'].isin(bad_dates)].copy()
        if excl:
            cohort_users = cohort_users.loc[~cohort_users['__uid__'].isin(excl)].copy()
        if cohort_users.empty:
            continue

        _, caps = get_trimmed_for_variant(cohort_users, daily_user_cums, e, method, pct)

        denom_w = (
            cohort_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                        .nunique().reset_index(name='N_users')
        )

        for k in range(start_k, e + 1):
            sum_prev = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 2, caps=caps
            ).rename(columns={'sum_cum': 'sum_prev'})
            sum_curr = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 1, caps=caps
            ).rename(columns={'sum_cum': 'sum_curr'})
            tmp = (
                denom_w
                .merge(sum_prev, on=['population', 'cost_date'])
                .merge(sum_curr, on=['population', 'cost_date'])
            )
            tmp['ARPU_prev']  = tmp['sum_prev'] / tmp['N_users']
            tmp['ARPU_curr']  = tmp['sum_curr'] / tmp['N_users']
            tmp['step_ratio'] = np.where(
                tmp['ARPU_prev'] > 0, tmp['ARPU_curr'] / tmp['ARPU_prev'], np.nan
            )
            mean_step, _, _ = weighted_mean_std_cv(
                tmp['step_ratio'].values, tmp['sum_prev'].values
            )
            step_rows.append({
                'day':             int(k),
                'growth_step':     float(mean_step),
                'effective_patch': f'{s}->{e}',
            })

    if not step_rows:
        return pd.DataFrame(cv_rows), pd.DataFrame()

    step_df = pd.DataFrame(step_rows).sort_values('day').reset_index(drop=True)

    first      = effective[0]
    fs, fe     = first['s'], first['e']
    excl_first = first['excluded_snapshot']
    base_end   = (as_of_date - pd.Timedelta(days=fe)).date()
    base_start = (as_of_date - pd.Timedelta(days=fe + (lookback_cohorts - 1))).date()

    base_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= base_start) &
        (u_base['cost_date'] <= base_end)
    ][['population', 'cost_date', '__uid__']].copy()
    bad_first = set(first['removed_dates'])
    if bad_first:
        base_users = base_users.loc[~base_users['cost_date'].isin(bad_first)].copy()
    if excl_first:
        base_users = base_users.loc[~base_users['__uid__'].isin(excl_first)].copy()

    _, base_caps = get_trimmed_for_variant(base_users, daily_user_cums, fe, method, pct)

    denom_base = (
        base_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                  .nunique().reset_index(name='N_users')
    )
    start_idx = 0 if fs == 1 else (fs - 1)
    sum_day1 = sum_cum_at_idx(
        daily_user_cums, cohort_users=base_users, idx=start_idx, caps=base_caps
    ).rename(columns={'sum_cum': 'sum_day1'})
    base = denom_base.merge(sum_day1, on=['population', 'cost_date'], how='inner')
    pooled_arpu_1 = (
        base['sum_day1'].sum() / base['N_users'].sum()
        if base['N_users'].sum() > 0 else 0.0
    )

    start_day = 1 if fs == 1 else fs
    out_rows  = [{'day': start_day,
                  'ARPU_nominal': float(pooled_arpu_1),
                  'growth_step':  np.nan,
                  'effective_patch': f'{fs}->{fe}'}]
    arpu = float(pooled_arpu_1)
    for _, row in step_df.iterrows():
        g = row['growth_step']
        if not np.isfinite(g):
            continue
        arpu *= float(g)
        out_rows.append({'day': int(row['day']),
                         'ARPU_nominal': float(arpu),
                         'growth_step':  float(g),
                         'effective_patch': row['effective_patch']})

    curve = pd.DataFrame(out_rows).sort_values('day').reset_index(drop=True)
    all_days = pd.DataFrame({'day': range(start_day, 366)})
    curve    = all_days.merge(curve, on='day', how='left')
    curve['effective_patch'] = curve['effective_patch'].ffill()
    curve['ARPU_nominal']    = curve['ARPU_nominal'].interpolate(
        method='linear', limit_area='inside'
    )
    curve = curve.dropna(subset=['ARPU_nominal']).reset_index(drop=True)
    curve['is_extrapolated'] = False

    return pd.DataFrame(cv_rows), curve


print('Persistent-trim curve builder defined (LS).')

In [ ]:
def variant_label(method, pct):
    return f'{method}_{round(pct * 100, 4):g}pct'


print(f'as_of_date = {AS_OF_DATE.date()}')
print('Building base tables for all populations...')

u_pop, daily_pop = build_user_revenue_cums(users_df, revenue_df, max_day=365)

diagnostics_rows = []
curves_dfs       = []

for population in POPULATIONS:
    variants = TRIM_VARIANTS.get(population, [])
    if not variants:
        print(f'\n\u26a0 No TRIM_VARIANTS defined for {population} \u2014 skipping.')
        continue

    print(f'\n{"#" * 70}')
    print(f'# POPULATION: {population}')
    print(f'{"#" * 70}')

    for (method, pct) in variants:
        label = variant_label(method, pct)
        print(f'\n{"=" * 60}')
        print(f'VARIANT: {label}  (method={method}, pct={pct})')
        print(f'{"=" * 60}')

        cv_rows_df, curve = build_curve_variant(
            u_pop, daily_pop,
            population=population, as_of_date=AS_OF_DATE,
            method=method, pct=pct, debug=True,
        )

        if not cv_rows_df.empty:
            cv_rows_df = cv_rows_df.copy()
            cv_rows_df['variant_label'] = label
            cv_rows_df['method']        = method
            cv_rows_df['pct']           = pct
            diagnostics_rows.append(cv_rows_df)

        if not curve.empty:
            curve = curve.copy()
            curve['variant_label'] = label
            curve['method']        = method
            curve['pct']           = pct
            curve['population']    = population
            curves_dfs.append(curve)

diagnostics_df = (
    pd.concat(diagnostics_rows, ignore_index=True)
    if diagnostics_rows else pd.DataFrame()
)
curves_df = (
    pd.concat(curves_dfs, ignore_index=True)
    if curves_dfs else pd.DataFrame()
)

if not diagnostics_df.empty:
    cols_order = [
        'population', 'variant_label', 'method', 'pct',
        'patch', 'cohort_start', 'cohort_end',
        'n_cohort_dates_total', 'n_cohort_dates_kept',
        'n_users_excluded_prior',
        'n_users_pre_trim', 'n_users_post_trim', 'n_users_dropped_by_trim',
        'total_rev_before_trim', 'total_rev_after_trim',
        'cv_before', 'cv_after', 'mean_after',
        'flagged', 'removed_dates',
    ]
    diagnostics_df = diagnostics_df[
        [c for c in cols_order if c in diagnostics_df.columns]
    ]

print(f'\ndiagnostics_df: {len(diagnostics_df):,} rows')
print(f'curves_df:      {len(curves_df):,} rows')

In [ ]:
if diagnostics_df.empty:
    print('No diagnostics to display.')
else:
    patch_order = [f'{s}->{e}' for (s, e) in PATCHES]

    for population in POPULATIONS:
        pop_diag = diagnostics_df.loc[diagnostics_df['population'] == population]
        if pop_diag.empty:
            continue

        variant_order = pop_diag['variant_label'].drop_duplicates().tolist()

        print(f'\n{"#" * 90}')
        print(f'# {population} \u2014 CV_AFTER per patch x variant')
        print(f'{"#" * 90}')
        cv_pivot = (
            pop_diag
            .pivot(index='patch', columns='variant_label', values='cv_after')
            .reindex(patch_order)[variant_order]
        )
        print(cv_pivot.to_string(float_format=lambda v: f'{v:.4f}'))

        print(f'\n{"-" * 90}')
        print(f'{population} \u2014 FLAGGED per patch x variant (True = cv_after > CV_THRESHOLD)')
        print(f'{"-" * 90}')
        flag_pivot = (
            pop_diag
            .pivot(index='patch', columns='variant_label', values='flagged')
            .reindex(patch_order)[variant_order]
        )
        print(flag_pivot.to_string())

In [ ]:
milestones = [1, 7, 14, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]

if curves_df.empty:
    print('No curves to display.')
else:
    for population in POPULATIONS:
        pop_curves = curves_df.loc[curves_df['population'] == population]
        if pop_curves.empty:
            continue

        variant_order = pop_curves['variant_label'].drop_duplicates().tolist()
        ms = pop_curves.loc[pop_curves['day'].isin(milestones)].copy()

        if ms.empty:
            continue

        ms_pivot = (
            ms.pivot(index='day', columns='variant_label', values='ARPU_nominal')
              .reindex(milestones)[variant_order]
        )
        print(f'\n{"#" * 90}')
        print(f'# {population} \u2014 ARPU_nominal at milestone days')
        print(f'{"#" * 90}')
        print(ms_pivot.to_string(float_format=lambda v: f'{v:,.4f}'))

        if len(variant_order) > 1:
            baseline_label = variant_order[0]
            print(f'\n{"-" * 90}')
            print(f'{population} \u2014 ARPU ratio vs {baseline_label}')
            print(f'{"-" * 90}')
            baseline = ms_pivot[baseline_label]
            ratios = ms_pivot.div(baseline, axis=0)
            print(ratios.to_string(float_format=lambda v: f'{v:.4f}'))

In [ ]:
diag_out = diagnostics_df.copy()
if 'removed_dates' in diag_out.columns:
    diag_out['removed_dates'] = diag_out['removed_dates'].apply(
        lambda lst: ','.join(str(d) for d in lst) if isinstance(lst, list) else ''
    )

diag_out.to_csv('lonestar_per_pop_persistent_trim_diagnostics.csv', index=False)
curves_df.to_csv('lonestar_per_pop_persistent_trim_curves.csv', index=False)

print('Saved:')
print('  lonestar_per_pop_persistent_trim_diagnostics.csv')
print('  lonestar_per_pop_persistent_trim_curves.csv')

from google.colab import files
files.download('lonestar_per_pop_persistent_trim_diagnostics.csv')
files.download('lonestar_per_pop_persistent_trim_curves.csv')